**Note**: This notebook should take around **three minutes** to run.

For a more comprehensive and detailed treatment of this problem, please refer to the [GitHub repository](https://github.com/HighDimensionalEconLab/symmetry_dynamic_programming).

# The Economic Problem:


#### Importing Packages

In [1]:
import numpy as np
import quantecon
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib import cm

In [2]:
torch.manual_seed(123) # fixing the seed

#### Setting Up the Economic Parameters 


In [3]:
N = 128 # Number of firms
alpha_0 = 1.0 # price coefficient
alpha_1 = 1.0 # price coefficient
beta = 0.95 # Discount factor
gamma = 90.0 # Adjustment cost coefficient 
sigma = 0.005 # Idiosyncratic shock standard deviation 
delta =  0.05 # Depreciation rate
eta = 0.001 # Aggeregate shock standard deviation
nu  = 1.0 # The demand curvature

### Setting Up the Grid ($\mathcal{X}_{\text{train}}$) and the DataLoader
1. Generating an intial condition for the economy: $X_0$
2. Given the initial condition $X_0$, and initial guess $u_0$, realizations of idiosyncratic and aggregate shocks, generating some trajectories of length $T$ to use for $\mathcal{X}_{\text{train}}$  

3. We use a linear policy for initial simulation: $u_0(X) = h_0 + \frac{h_1}{N}\sum_{i=1}^N X_i$ 

3.a) $h_0>0$ and $h_1<0$ guarantees stationarity and positivity. 

3.b) $|\frac{h_0}{h_1}|<1$ guarantees the positivity of the prices in the sample, i.e., $p(X)>0$

In [4]:
# The initial state of the economy: X_0
X_0_loc =  0.9
X_0_scale = 0.05
X_0 = torch.normal(X_0_loc, X_0_scale, size=(N,)).abs()

In [5]:
# Generating the initial simulated path
T = 63 # Length of the paths or trajectories
num_trajectories = 16 # Number of trajectories/path generated

def u_0(X): # The initial guess to generate the trajectories
    return 0.2 - 0.3*X.mean(1, keepdim=True) 
 

In [6]:
# Creating the idiosyncratic and aggregate shocks
w = torch.randn(num_trajectories,T,N,) #idiosyncratic shocks
omega = torch.randn(num_trajectories,T,1,) #aggregate shocks

In [7]:
data = torch.zeros(num_trajectories,T+1 ,N)

data[:, 0, :] = X_0
for t in range(T):
    data[:, t + 1, :] = u_0(data[:, t, :]) + (1 - delta) * data[:, t, :] + sigma* w[:,t,:]+ eta * omega[:, t]

data = data.flatten(start_dim=0, end_dim=1) # Turning the matrix into a tensor

In [8]:
data_loader = DataLoader(data, batch_size=16, shuffle= True)

#### Setting up the Neural Network

Here we use the $\Phi(\text{ReLU})$ structure, for the `moment-based` structure refer to this [code](https://github.com/HighDimensionalEconLab/econ_layers/blob/982f5658064a3f6efcd53d309d154b443cd48ba7/econ_layers/layers.py#L216-L254).

This is an approximation for the invetment/adjustment policy $\hat{u}: \mathbb{R}^N \rightarrow \mathbb{R}$, where $N$ is the number of agents.

The symmetry induces a structure on the design of the neural network:

$\begin{align}
\hat{u}(X) = \rho\big(\frac{1}{N}\sum_{i=1}^N\phi(X_i)\big)
\end{align}$


where $\phi:\mathbb{R}\rightarrow \mathbb{R}^L$ and $\rho:\mathbb{R}^L\rightarrow \mathbb{R}$

Both $\rho$ and $\phi$ are going to be represented with a neural netwrok.


In [9]:
L = 4

In [10]:
class U_hat_NN(nn.Module):
    def __init__(self,
                 dim_hidden = 128,):
        super().__init__()
        
        self.rho = nn.Sequential(
            nn.Linear(L, dim_hidden),
            nn.ReLU(),
            nn.Linear(dim_hidden, dim_hidden, bias=True),
            nn.ReLU(),
            nn.Linear(dim_hidden, dim_hidden, bias=True),
            nn.ReLU(),
            nn.Linear(dim_hidden, dim_hidden, bias=True),
            nn.ReLU(),
            nn.Linear(dim_hidden, 1),
        )
        
        self.phi = nn.Sequential(
            nn.Linear(1, dim_hidden),
            nn.ReLU(),
            nn.Linear(dim_hidden, L, bias=True),
        )
        
        
    def forward(self, X):
        num_batches, N = X.shape
        phi_X = torch.stack(
            [torch.mean(self.phi(X[i, :].reshape([N, 1])), 0) for i in range(num_batches)]
        )
        return self.rho(phi_X)

#### Initializing the Neural Network and Defining the Optimizer

In [11]:
u_hat= U_hat_NN()
learning_rate = 1e-3
optimizer = torch.optim.Adam(u_hat.parameters(), lr=learning_rate) #Adam Optimizer 
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.8) #dampening the learning rate

In [12]:
num_epochs = 61 # step of optimization
print_epoch_frequency = 10 # how often printing the results

### Implementing the Economics 
Complete it later
1. Euler residuals
2. One draw of aggregate shocks
3. Gaussian quadratures for calculating the expectations with respect to aggregate shocks

In [13]:
# One draw of idiosyncratic shocks to calcualte expectation wrt to idiosyncratic shocks
one_draw_idio_vec = torch.randn(1, N) 
one_draw_idio = (one_draw_idio_vec - one_draw_idio_vec.mean()) / one_draw_idio_vec.std() #Normalizing to make sure it is mean zero


In [14]:
# Gaussian quadrature nodes to calculate expectation wrt to aggregate shocks
omega_quadrature_nodes = 7 # number of nodes
nodes, weights = quantecon.quad.qnwnorm(omega_quadrature_nodes)
quadrature_nodes = torch.tensor(nodes, dtype=torch.float32)
quadrature_weights = torch.tensor(weights, dtype=torch.float32)

In [15]:
def residuals(batch):
    X = batch
    
    #u(X)
    u_X = u_hat(X)
    
    # X'
    X_primes = torch.stack([u_X + (1 - delta) * X + sigma * one_draw_idio + eta * node for node in quadrature_nodes]).type_as(X)
    
    #p(X'): price next period
    p_primes = alpha_0 - alpha_1 * X_primes.pow(nu).mean(2)
    
    #E[p(X')]
    Ep = (p_primes.T @ quadrature_weights).type_as(X).reshape(-1, 1)
    
    #E[u(X')]
    Eu = ((torch.stack(tuple(u_hat(X_primes[i]) for i in range(len(quadrature_nodes)))).squeeze(2).T@ quadrature_weights)
          .type_as(X).reshape(-1, 1))
    
    # Euler residuals
    residuals = gamma * u_X - beta * (Ep + gamma * (1 -delta) * Eu) 
    
    return residuals

### Training Loop (Optimization Process)
Finish it later

Here, we minimize the mean squared of Euler residuals over the grid points:


where $\theta$ is the coefficients of the neural network, and the optimization id done with gradient-based optimizers such as `Gradient Descent (GD)` and `Adam`. 

In [16]:
for epoch in range(num_epochs):
    for batch in data_loader:
        
        optimizer.zero_grad() # Resetting the gradients (with respect to NN coefficients)
        
        Euler_res = residuals(batch) # Constructing the Euler and initial condition residuals
        
        loss = Euler_res.pow(2).mean() # Calculating the mean squared (over the grid points) of Euler residuals
        
        
        loss.backward() # Calculating the gradients (with respect to NN coefficients)
        optimizer.step() # Using the gradients to minimize the loss function
        
    scheduler.step() # Dampening the learning rate
    
    if epoch % print_epoch_frequency == 0:
        print(f"epoch = {epoch}, loss = {loss.detach().numpy():.2e}")


epoch = 0, loss = 8.95e-03
epoch = 10, loss = 5.59e-07
epoch = 20, loss = 2.81e-07
epoch = 30, loss = 4.39e-07
epoch = 40, loss = 1.95e-05
epoch = 50, loss = 1.91e-07
epoch = 60, loss = 1.09e-06
